In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")

# Install FAISS-CPU
!pip install -q faiss-cpu numpy tqdm

CUDA available: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 93.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/'
OUTPUT_DIR = '/content/retrieval'

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [3]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
from dataclasses import dataclass
import time

import faiss
print(f"✓ FAISS version: {faiss.__version__}")
print(f"  NumPy version: {np.__version__}")

import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")

✓ FAISS version: 1.13.2
  NumPy version: 2.0.2

PyTorch CUDA available: True


In [4]:
# Model name
MODEL_NAME = "sphilberta"

# Retrieval configuration
K_VALUES = [5, 10, 20]  # Top-k values to retrieve
SIMILARITY_THRESHOLD = 0.5  # Minimum cosine similarity

In [9]:
# ============================================================================
# CELL 5: Load Embeddings and IDs (JSON VERSION)
# ============================================================================

def load_embeddings(embeddings_dir: str, model_name: str) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict]:
    """
    Load embeddings and chunk IDs.

    Returns:
        (bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata)
    """
    print("Loading embeddings and IDs...\n")

    # Load embeddings
    bdc_emb_path = f"{embeddings_dir}/bdc_embeddings_{model_name}.npy"
    psc_emb_path = f"{embeddings_dir}/psc_embeddings_{model_name}.npy"

    print(f"Loading BDC embeddings from: {Path(bdc_emb_path).name}")
    bdc_embeddings = np.load(bdc_emb_path)
    print(f"  Shape: {bdc_embeddings.shape}")

    print(f"Loading PSC embeddings from: {Path(psc_emb_path).name}")
    psc_embeddings = np.load(psc_emb_path)
    print(f"  Shape: {psc_embeddings.shape}\n")

    # Load IDs from JSON (single array, not line-by-line)
    bdc_ids_path = f"{embeddings_dir}/bdc_ids_{model_name}.json"
    psc_ids_path = f"{embeddings_dir}/psc_ids_{model_name}.json"

    print(f"Loading BDC IDs from: {Path(bdc_ids_path).name}")
    with open(bdc_ids_path, 'r') as f:
        bdc_ids = json.load(f)  # Load entire JSON array
    print(f"  Loaded {len(bdc_ids):,} IDs")

    print(f"Loading PSC IDs from: {Path(psc_ids_path).name}")
    with open(psc_ids_path, 'r') as f:
        psc_ids = json.load(f)  # Load entire JSON array
    print(f"  Loaded {len(psc_ids):,} IDs\n")

    # Load metadata
    metadata_path = f"{embeddings_dir}/metadata_{model_name}.json"
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    print(f"✓ Loaded metadata\n")

    # Verification
    assert bdc_embeddings.shape[0] == len(bdc_ids), "BDC embeddings and IDs mismatch!"
    assert psc_embeddings.shape[0] == len(psc_ids), "PSC embeddings and IDs mismatch!"
    assert bdc_embeddings.shape[1] == psc_embeddings.shape[1], "Embedding dimensions mismatch!"

    print("All verification checks passed")

    return bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata


# Load data
bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata = load_embeddings(
    DATA_DIR,
    MODEL_NAME
)

print(f"\nLoaded:")
print(f"  BDC queries: {len(bdc_ids):,}")
print(f"  PSC candidates: {len(psc_ids):,}")
print(f"  Embedding dimension: {bdc_embeddings.shape[1]}")

Loading embeddings and IDs...

Loading BDC embeddings from: bdc_embeddings_sphilberta.npy
  Shape: (12730, 768)
Loading PSC embeddings from: psc_embeddings_sphilberta.npy
  Shape: (13179, 768)

Loading BDC IDs from: bdc_ids_sphilberta.json
  Loaded 12,730 IDs
Loading PSC IDs from: psc_ids_sphilberta.json
  Loaded 13,179 IDs

✓ Loaded metadata

All verification checks passed

Loaded:
  BDC queries: 12,730
  PSC candidates: 13,179
  Embedding dimension: 768


In [14]:
# build index
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """
    Build FAISS index for efficient similarity search.

    Uses IndexFlatIP (Inner Product) for exact cosine similarity search
    on normalized embeddings.

    Args:
        embeddings: Normalized embeddings [num_items, embedding_dim]

    Returns:
        FAISS index
    """
    embedding_dim = embeddings.shape[1]
    num_vectors = embeddings.shape[0]

    print(f"Building FAISS index...")
    print(f"  Index type: IndexFlatIP (exact cosine similarity)")
    print(f"  Embedding dimension: {embedding_dim}")
    print(f"  Number of vectors: {num_vectors:,}")
    print(f"  Device: CPU")

    # Create index (Inner Product for normalized vectors = cosine similarity)
    index = faiss.IndexFlatIP(embedding_dim)

    # Add vectors to index
    print(f"Adding vectors to index...")
    index.add(embeddings.astype('float32'))

    print(f"Index built successfully")
    print(f"Total vectors in index: {index.ntotal:,}\n")

    return index


# Build index for PSC candidates
psc_index = build_faiss_index(psc_embeddings)

Building FAISS index...
  Index type: IndexFlatIP (exact cosine similarity)
  Embedding dimension: 768
  Number of vectors: 13,179
  Device: CPU
Adding vectors to index...
Index built successfully
Total vectors in index: 13,179



In [15]:
def retrieve_top_k(
    query_embeddings: np.ndarray,
    index: faiss.Index,
    k: int,
    query_ids: List[str],
    candidate_ids: List[str],
    threshold: float = 0.0
) -> List[Dict]:
    """
    Retrieve top-k candidates for each query.

    Args:
        query_embeddings: Query embeddings [num_queries, dim]
        index: FAISS index containing candidate embeddings
        k: Number of top candidates to retrieve
        query_ids: List of query chunk IDs
        candidate_ids: List of candidate chunk IDs
        threshold: Minimum similarity threshold (optional)

    Returns:
        List of retrieval results, one per query
    """
    print(f"Retrieving top-{k} candidates for {len(query_ids):,} queries...")

    # Search index
    start_time = time.time()
    similarities, indices = index.search(query_embeddings.astype('float32'), k)
    search_time = time.time() - start_time

    print(f"✓ Search complete in {search_time:.2f}s")
    print(f"  Average time per query: {search_time/len(query_ids)*1000:.2f}ms\n")

    # Build results
    results = []
    queries_below_threshold = 0

    for i, query_id in enumerate(tqdm(query_ids, desc="Building results")):
        top_k_indices = indices[i]
        top_k_similarities = similarities[i]

        # Filter by threshold if specified
        candidates = []
        max_similarity = float(top_k_similarities[0]) if len(top_k_similarities) > 0 else 0.0

        for idx, sim in zip(top_k_indices, top_k_similarities):
            if sim >= threshold:
                candidates.append({
                    "candidate_id": candidate_ids[idx],
                    "similarity": float(sim),
                    "rank": len(candidates) + 1
                })

        # Check if max similarity is below threshold
        if max_similarity < threshold:
            queries_below_threshold += 1

        results.append({
            "query_id": query_id,
            "max_similarity": max_similarity,
            "num_candidates": len(candidates),
            "candidates": candidates,
            "below_threshold": max_similarity < threshold
        })

    print(f"\nRetrieval statistics:")
    print(f"  Queries with max similarity < {threshold}: {queries_below_threshold} ({queries_below_threshold/len(query_ids)*100:.1f}%)")
    print(f"  Average candidates per query: {np.mean([len(r['candidates']) for r in results]):.2f}")

    return results

In [16]:
print("="*60)
print("RUNNING RETRIEVAL FOR ALL K VALUES")
print("="*60 + "\n")

retrieval_results = {}

for k in K_VALUES:
    print(f"\n{'='*60}")
    print(f"K = {k}")
    print(f"{'='*60}\n")

    results = retrieve_top_k(
        bdc_embeddings,
        psc_index,
        k,
        bdc_ids,
        psc_ids,
        threshold=SIMILARITY_THRESHOLD
    )

    retrieval_results[k] = results

    # Save results as JSONL (one query result per line)
    output_path = f"{OUTPUT_DIR}/retrieval_results_{MODEL_NAME}_k{k}.jsonl"
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')

    print(f"✓ Saved {len(results):,} results to: {Path(output_path).name}")

print(f"\n{'='*60}")
print("ALL RETRIEVAL COMPLETE")
print(f"{'='*60}")

RUNNING RETRIEVAL FOR ALL K VALUES


K = 5

Retrieving top-5 candidates for 12,730 queries...
✓ Search complete in 4.13s
  Average time per query: 0.32ms



Building results:   0%|          | 0/12730 [00:00<?, ?it/s]


Retrieval statistics:
  Queries with max similarity < 0.5: 2 (0.0%)
  Average candidates per query: 5.00
✓ Saved 12,730 results to: retrieval_results_sphilberta_k5.jsonl

K = 10

Retrieving top-10 candidates for 12,730 queries...
✓ Search complete in 3.71s
  Average time per query: 0.29ms



Building results:   0%|          | 0/12730 [00:00<?, ?it/s]


Retrieval statistics:
  Queries with max similarity < 0.5: 2 (0.0%)
  Average candidates per query: 9.99
✓ Saved 12,730 results to: retrieval_results_sphilberta_k10.jsonl

K = 20

Retrieving top-20 candidates for 12,730 queries...
✓ Search complete in 3.25s
  Average time per query: 0.26ms



Building results:   0%|          | 0/12730 [00:00<?, ?it/s]


Retrieval statistics:
  Queries with max similarity < 0.5: 2 (0.0%)
  Average candidates per query: 19.98
✓ Saved 12,730 results to: retrieval_results_sphilberta_k20.jsonl

ALL RETRIEVAL COMPLETE


In [17]:
retrieval_metadata = {
    "model_name": MODEL_NAME,
    "embedding_metadata": metadata,
    "retrieval_config": {
        "k_values": K_VALUES,
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "index_type": "IndexFlatIP"
    },
    "statistics": {
        "num_queries": len(bdc_ids),
        "num_candidates": len(psc_ids),
        "embedding_dim": bdc_embeddings.shape[1]
    },
    "results_per_k": {}
}

# Add statistics for each k
for k in K_VALUES:
    results = retrieval_results[k]

    queries_below_threshold = sum(1 for r in results if r["below_threshold"])
    avg_candidates = np.mean([len(r["candidates"]) for r in results])
    avg_max_similarity = np.mean([r["max_similarity"] for r in results])

    retrieval_metadata["results_per_k"][f"k{k}"] = {
        "k": k,
        "num_results": len(results),
        "queries_below_threshold": queries_below_threshold,
        "queries_below_threshold_pct": queries_below_threshold / len(results) * 100,
        "avg_candidates_per_query": float(avg_candidates),
        "avg_max_similarity": float(avg_max_similarity),
        "output_file": f"retrieval_results_{MODEL_NAME}_k{k}.jsonl"
    }

# Save metadata
metadata_path = f"{OUTPUT_DIR}/retrieval_metadata_{MODEL_NAME}.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(retrieval_metadata, f, indent=2, ensure_ascii=False)

In [18]:
print(f"\nModel: {MODEL_NAME}")
print(f"Queries: {len(bdc_ids):,}")
print(f"Candidates: {len(psc_ids):,}")

print(f"\nResults generated for k values: {K_VALUES}")

for k in K_VALUES:
    stats = retrieval_metadata["results_per_k"][f"k{k}"]
    print(f"\nK = {k}:")
    print(f"  Results: {stats['num_results']:,}")
    print(f"  Avg candidates per query: {stats['avg_candidates_per_query']:.2f}")
    print(f"  Avg max similarity: {stats['avg_max_similarity']:.4f}")
    print(f"  Queries below threshold: {stats['queries_below_threshold']} ({stats['queries_below_threshold_pct']:.1f}%)")
    print(f"  File: {stats['output_file']}")

print(f"\nOutput directory: {OUTPUT_DIR}")

# Verify files exist
print("\nVerification:")
output_files = [f"retrieval_results_{MODEL_NAME}_k{k}.jsonl" for k in K_VALUES]
output_files.append(f"retrieval_metadata_{MODEL_NAME}.json")

for filename in output_files:
    filepath = Path(OUTPUT_DIR) / filename
    exists = "✓" if filepath.exists() else "✗"
    size = filepath.stat().st_size / 1024**2 if filepath.exists() else 0
    print(f"  {exists} {filename} ({size:.2f} MB)")


Model: sphilberta
Queries: 12,730
Candidates: 13,179

Results generated for k values: [5, 10, 20]

K = 5:
  Results: 12,730
  Avg candidates per query: 5.00
  Avg max similarity: 0.7509
  Queries below threshold: 2 (0.0%)
  File: retrieval_results_sphilberta_k5.jsonl

K = 10:
  Results: 12,730
  Avg candidates per query: 9.99
  Avg max similarity: 0.7509
  Queries below threshold: 2 (0.0%)
  File: retrieval_results_sphilberta_k10.jsonl

K = 20:
  Results: 12,730
  Avg candidates per query: 19.98
  Avg max similarity: 0.7509
  Queries below threshold: 2 (0.0%)
  File: retrieval_results_sphilberta_k20.jsonl

Output directory: /content/retrieval

Verification:
  ✓ retrieval_results_sphilberta_k5.jsonl (8.81 MB)
  ✓ retrieval_results_sphilberta_k10.jsonl (16.04 MB)
  ✓ retrieval_results_sphilberta_k20.jsonl (30.55 MB)
  ✓ retrieval_metadata_sphilberta.json (0.00 MB)


In [19]:
# Show first 3 query results for k=10
k = 10
sample_results = retrieval_results[k][:3]

for i, result in enumerate(sample_results, 1):
    print(f"\nQuery {i}: {result['query_id']}")
    print(f"  Max similarity: {result['max_similarity']:.4f}")
    print(f"  Retrieved {result['num_candidates']} candidates:")

    for candidate in result['candidates'][:5]:  # Show top 5
        print(f"    {candidate['rank']}. {candidate['candidate_id']} (sim: {candidate['similarity']:.4f})")

    if result['num_candidates'] > 5:
        print(f"    ... and {result['num_candidates'] - 5} more")


Query 1: 10015_sent_0
  Max similarity: 0.7962
  Retrieved 10 candidates:
    1. 022_Hieronymus-Stridonensis_Epistolae_window_2989 (sim: 0.7962)
    2. 022_Hieronymus-Stridonensis_Epistolae_window_655 (sim: 0.7922)
    3. 004_Cyprianus-Carthaginensis_Epistolae-2_window_262 (sim: 0.7882)
    4. 031_Augustinus-Hipponensis_Confessiones_window_9 (sim: 0.7881)
    5. 033_Augustinus-Hipponensis_Epistolae_window_713 (sim: 0.7824)
    ... and 5 more

Query 2: 10015_sent_1
  Max similarity: 0.7579
  Retrieved 10 candidates:
    1. 033_Augustinus-Hipponensis_Epistolae_window_2951 (sim: 0.7579)
    2. 033_Augustinus-Hipponensis_Epistolae_window_4176 (sim: 0.7468)
    3. 004_Cyprianus-Carthaginensis_Epistolae-2_window_241 (sim: 0.7404)
    4. 004_Cyprianus-Carthaginensis_Epistolae-2_window_145 (sim: 0.7340)
    5. 033_Augustinus-Hipponensis_Epistolae_window_2468 (sim: 0.7330)
    ... and 5 more

Query 3: 10015_sent_2
  Max similarity: 0.6930
  Retrieved 10 candidates:
    1. 022_Hieronymus-Strido